# Analiza streszczeń literatury młodzieżowej

## Import bibliotek 

In [ ]:
import nltk                                                  # nltk is a leading platform for building Python programs to work with human language data
import json                                                  # json is a built-in Python module for working with JSON data
import pandas as pd                                          # pandas is a powerful data manipulation and analysis library for Python
import morfeusz2                                             # morfeusz2 is a Python library for morphological analysis of Polish language
import string                                                # String is a built-in Python module that provides a collection of string constants and utility functions for string manipulation
import operator as op                                        # Operator is a built-in Python module that provides a set of efficient functions for performing common operations on objects, such as arithmetic, comparison, and logical operations
import itertools as it                                       # itertools is a built-in Python module that provides a collection of tools for working with iterators, which are objects that can be iterated (looped) over
from nltk.corpus import PlaintextCorpusReader                # PlaintextCorpusReader is a class in the NLTK library that allows you to read and process plain text files as a corpus, which is a collection of texts used for linguistic analysis
from sklearn.feature_extraction.text import CountVectorizer  # CountVectorizer is a class in the scikit-learn library that converts a collection of text documents into a matrix of token counts, which can be used for various natural language processing tasks
from sklearn.feature_extraction.text import TfidfVectorizer  # TfidfVectorizer is a class in the scikit-learn library that converts a collection of text documents into a matrix of TF-IDF features, which can be used for various natural language processing tasks
from nltk.tokenize import word_tokenize                      # word_tokenize is a function in the NLTK library that splits a given text into individual words or tokens, based on whitespace and punctuation

In [ ]:
nltk.pathsec.ENFORCE = False   #Disable the path security check for nltk data files
nltk.download('punkt_tab')      # Download the punkt tokenizer data

## Utworzenie korpusu dokumentów

In [ ]:
corpus_dir = "./Literature"
corpus = PlaintextCorpusReader(corpus_dir, r".*\.txt")
files_names = corpus.fileids()

## Wstępne przygotowanie dokumentów (preprocessing)

In [ ]:
documents_dict = {}

# Read each document and store it in the dictionary
for file_name in files_names:
    documents_dict[file_name] = corpus.raw(file_name)
    
print(json.dumps(documents_dict, indent=4, ensure_ascii=False))

In [ ]:
lengths = {}

for file_name in documents_dict:
    lengths[file_name] = {
        "pre": len(word_tokenize(documents_dict[file_name])), # Count the number of words in the document
    }
print(json.dumps(lengths, indent=4, ensure_ascii=False))

In [ ]:
stopwords_file = open("./stopwords_pl.txt", "r", encoding="utf-8") 
stopwords = stopwords_file.read().splitlines()
stopwords = stopwords[4:]  # Skip the first four lines of the stopwords file
stopwords_file.close()
stopwords

In [ ]:
# Example usage of morfeusz2 for morphological analysis of Polish text
morfer = morfeusz2.Morfeusz()               
print(*morfer.analyse("Ala ma kota."), sep="\n")             

In [ ]:
def lemmatize_text(text):                                                            # lemmatization is the process of reducing words to their base or root form
    morfer = morfeusz2.Morfeusz()                                                    # Create an instance of the Morfeusz class
    segments = it.groupby(morfer.analyse(text), op.itemgetter(0,1))                  # Analyze the text and obtain morphological segments, then group them by their starting and ending positions
    
    def disambiguate(group):
        pairs = [(len(descr), lemma) for _, _, (_, lemma, descr, _, _) in group]     # Create a list of pairs containing the length of the description and the corresponding lemma for each segment in the group
        perpl, lemma = min(pairs)                                                    # Find the pair with the minimum length of description (i.e., the most specific lemma)
        return lemma.split(":")[0] if lemma else None                                # Return the lemma without any additional information (if available), otherwise return None
    
    lemmas = [disambiguate(group) for _, group in segments]                          # Apply the disambiguation function to each group of segments and collect the resulting lemmas into a list
    return " ".join(filter(str.isalpha, lemmas))                                     # Join the lemmatized words into a single string, filtering out any non-alphabetic tokens
    
    


In [ ]:
for file_name in documents_dict:
    documents_dict[file_name] = documents_dict[file_name].lower()
    documents_dict[file_name] = "".join([char for char in documents_dict[file_name] if char not in string.punctuation])                           # Remove punctuation from the text
    documents_dict[file_name] = lemmatize_text(documents_dict[file_name])                                                                         # Lemmatize the text using the lemmatize_text function
    documents_dict[file_name] = " ".join([word for word in word_tokenize(documents_dict[file_name], language='polish') if word not in stopwords])  # Remove stopwords from the text using the list of stopwords
print(json.dumps(documents_dict, indent=4, ensure_ascii=False))

In [ ]:
for file_name in documents_dict:
    lengths[file_name]['post'] = len(word_tokenize(documents_dict[file_name]))                  # Count the number of words in the document after preprocessing
print(json.dumps(lengths, indent=4, ensure_ascii=False))

In [ ]:
lengths = pd.DataFrame.from_dict(lengths, orient='index')
lengths['change'] = lengths['post'] - lengths['pre']
lengths['change_percent'] = ((lengths['change'] / lengths['pre']) * 100).round(2)
lengths


## Utworzenie macierzy częstości (word frequency matrix) 

In [ ]:
documents_df = pd.DataFrame.from_dict(documents_dict, orient='index', columns=['context'])
documents_df

In [ ]:
matrix_tf = CountVectorizer().fit_transform(documents_df['context'])        # Create a term frequency matrix using CountVectorizer
matrix_tfidf = TfidfVectorizer().fit_transform(documents_df['context'])     # Create a TF-IDF matrix using TfidfVectorizer

In [ ]:
sparsity_tf = (1-(matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1])) * 100)     # Calculate the sparsity of the term frequency matrix
sparsity_tfidf = (1-(matrix_tfidf.nnz / (matrix_tfidf.shape[0] * matrix_tfidf.shape[1])) * 100)  # Calculate the sparsity of the TF-IDF matrix